# Pico – large pre-training + grammar fine-tuning on Kaggle

**One-time setup**
1. Verify your phone number in Kaggle settings (needed for GPU + internet).
2. Upload your `train.jsonl` as a Kaggle Dataset (**Datasets → New Dataset**), then in this notebook **Add Input → your dataset**.
3. In the notebook sidebar, under **Session options**: set **Accelerator → GPU T4 x2** and turn **Internet → On**.

**Run it in the background:** click **Save Version → Save & Run All (Commit)**. It keeps running with your browser closed (up to 12 hours), then the results appear in the version's **Output** tab.

**If pre-training didn't finish in one session** (the log says *"Time limit reached"*): open the notebook, **Add Input → Notebook Output → this notebook's latest version**, and *Save & Run All* again. It picks up the data and checkpoint from the previous run and continues.

**Re-train only the grammar model** (pre-training already finished): **Add Input → Your Work → Notebooks → this notebook** (the version whose log says *"Pre-training done"*), then *Save & Run All*. It skips data preparation and pre-training and only runs the grammar fine-tuning (~1–1.5 hours).

## 1. Setup

In [ ]:
import os, glob, shutil, time
START = time.time()          # Kaggle sessions end after 12 hours
W = '/kaggle/working'        # everything saved here becomes the notebook's output
!nvidia-smi
if not os.path.exists('/tmp/Pico'):
    !git clone https://github.com/Abhinav-Krishna-P/Pico-Next_Token_Predictor.git /tmp/Pico
%cd /tmp/Pico
!pip install -q sentencepiece datasets

## 2. Restore the previous run (if added as input)

In [ ]:
def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    return hits[0] if hits else None

# the grammar model is not restored: it is always fine-tuned again
for name in ['pico_big.pt', 'pico_big_tokenizer.model']:
    src = find(name)
    if src and not os.path.exists(f'{W}/{name}'):
        shutil.copy(src, W)
        print('restored', name)

import torch
PRETRAIN_DONE = os.path.exists(f'{W}/pico_big.pt') and     torch.load(f'{W}/pico_big.pt', map_location='cpu').get('finished', False)
print('pre-training finished:', PRETRAIN_DONE)

prev = find('data_big/train.bin')
if prev and not PRETRAIN_DONE and not os.path.exists(f'{W}/data_big/train.bin'):
    shutil.copytree(os.path.dirname(prev), f'{W}/data_big')
    print('restored prepared data')


## 3. Prepare the data (first run only, ~30–60 min)
Full TinyStories + 300k FineWeb-Edu documents → 16k tokenizer → ~965M tokens.

In [ ]:
if PRETRAIN_DONE:
    print('Pre-training already finished, data not needed')
elif not os.path.exists(f'{W}/data_big/train.bin'):
    !python -m training.prepare_data --out-dir {W}/data_big
    for f in glob.glob(f'{W}/data_big/raw_*.txt'):
        os.remove(f)     # raw text is no longer needed; keeps the output small
else:
    print('Data already prepared')


## 4. Pre-train (~34M parameters, 600M tokens)
Stops by itself before the 12-hour limit (keeping ~1 hour for fine-tuning), saving a checkpoint you can continue from.

In [ ]:
if PRETRAIN_DONE:
    print('Pre-training already finished')
else:
    hours_left = 11.0 - (time.time() - START) / 3600
    limit = max(0.25, hours_left - 1.5)
    LIMIT = f'{limit:.2f}'   # shell commands can't use format specs like {limit:.2f}
    print(f'Pre-training time budget: {limit:.1f} hours')
    !python -m training.pretrain --data-dir {W}/data_big --out {W}/pico_big.pt --time-limit-hours {LIMIT}


## 5. Try the pre-trained model

In [ ]:
import torch, sentencepiece as spm
from pico import Pico
ckpt = torch.load(f'{W}/pico_big.pt', map_location='cuda')
PRETRAIN_DONE = ckpt.get('finished', False)
sp = spm.SentencePieceProcessor(model_file=f'{W}/pico_big_tokenizer.model')
model = Pico(**ckpt['config']).cuda().eval()
model.load_state_dict(ckpt['model'])
print(f"step {ckpt['step']}, val loss {ckpt['val_loss']:.4f}, finished: {PRETRAIN_DONE}")
print()
for prompt in ['Once upon a time', 'The history of science', 'My favourite food is']:
    idx = torch.tensor([sp.encode(prompt)], device='cuda')
    print(sp.decode(model.generate(idx, max_new_tokens=100, stop_id=sp.eos_id())[0].tolist()))
    print()
del model, ckpt
torch.cuda.empty_cache()

## 6. Fine-tune for grammar correction (~1 hour)
Runs once pre-training has finished. Trains on:
- your `train.jsonl` (100k pairs)
- ~385k short synthetic pairs with common learner mistakes (*a apple*, *he dont*, *she go yesterday* …) from `training/make_synthetic.py`
- the [English Practical Grammar dataset](https://github.com/Abhinav-Krishna-P/English-practical-grammer-Dataset): **all 4,233 conversations** (as *Question → Answer*, x10)
  plus ~650 correction pairs extracted from its exercises by `training/extract_grammar_pairs.py` (x20)

The context grows to 1,024 tokens so that even the longest answers fit (nothing is skipped).


In [ ]:
data = find('train.jsonl')
if not PRETRAIN_DONE:
    print('Pre-training not finished yet: add this notebook output as input and run again')
elif data is None:
    print('train.jsonl not found: add your dataset as input (Add Input)')
else:
    if not os.path.exists('/tmp/synthetic.jsonl'):
        !timeout 3600 python -m training.make_synthetic --out /tmp/synthetic.jsonl --pairs 400000
    if not os.path.exists('/tmp/synthetic.jsonl'):
        raise RuntimeError('make_synthetic.py failed (see the error above), not fine-tuning')

    # English Practical Grammar dataset: every conversation + correction pairs from its exercises
    if not os.path.exists('/tmp/epg'):
        !git clone -q --depth 1 https://github.com/Abhinav-Krishna-P/English-practical-grammer-Dataset /tmp/epg
    !python -m training.extract_grammar_pairs --src /tmp/epg --out {W}/grammar_book.jsonl
    BOOK = ' '.join(f'"{f}*10"' for f in sorted(glob.glob('/tmp/epg/data/*.jsonl')))
    print(len(glob.glob('/tmp/epg/data/*.jsonl')), 'dataset files')

    !python -m training.finetune_grammar --data {data} /tmp/synthetic.jsonl "{W}/grammar_book.jsonl*20" {BOOK} --base {W}/pico_big.pt --out {W}/pico_big_grammar.pt --tokenizer-out {W}/pico_big_grammar_tokenizer.model --block-size 1024 --steps 20000 --batch-size 32


## 7. Test + export a small file for your laptop

In [ ]:
if os.path.exists(f'{W}/pico_big_grammar.pt'):
    !python -m training.export_model {W}/pico_big_grammar.pt {W}/pico_big_grammar_small.pt
    for s in ['they was playing in park', 'She go to school yesterday.', 'I has a apple.',
              'He dont like coffee.', 'I am work as software engineer.', 'My brother are taller then me.',
              'She bought a new furnitures.', 'I look forward to meet you.']:
        print('IN :', s)
        !python correct.py --model {W}/pico_big_grammar_small.pt "{s}"
        print()
    for q in ['What is a noun?', 'When do we use "an" instead of "a"?']:
        print('Q :', q)
        !python ask.py --model {W}/pico_big_grammar_small.pt "{q}"
        print()
    # full benchmark report (download report.html from the Output tab)
    !python -m evaluation.evaluate --model {W}/pico_big_grammar_small.pt --baseline none --pretrain {W}/pico_big.pt --overlap {W}/grammar_book_test_overlap.json --out {W}/report.html --runs 1


## 8. Use it on your laptop

From this version's **Output** tab download **`pico_big_grammar_small.pt`**, **`pico_big_grammar_tokenizer.model`** and **`grammar_book_test_overlap.json`** into the `models/` folder of the Pico project, then:

```
python correct.py "they was playing in park"
python ask.py "What is a noun?"
```